## Project: Labels vs. Scale: Breaking Even on Banking77 with Encoders and Decoders

#### Frontier model zero-shot control

This notebook runs **Claude Opus 5.5** zero-shot on the validation (385) and test (3,080) sets, with the **same prompt and the same matching rule as Qwen zero-shot**. It writes two prediction files in the shared format (`frontier_zeroshot_seed0_val.csv` and `frontier_zeroshot_seed0_test.csv`) into `results/` (copied to the shared `results` folder on Drive).

Run order: Setup, Smoke test (5 examples), Full validation run, Full test run. Progress is cached on disk, so a stopped run resumes where it left off.

#### Setup

In [9]:
# Install the Anthropic SDK
!pip install -q anthropic pyarrow scikit-learn

#### Paths and code

The code and data live on Drive and are copied to Colab's local disk at the start of every session. Prediction files are saved to the same `results` folder on Drive as the other models.

In [10]:
import os, sys, shutil

from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/banking77"                      # Local copy of the repo (lost when the session ends)
DRIVE_DIR = "/content/drive/MyDrive/banking77"      # Permanent folder on Drive
RESULTS = f"{DRIVE_DIR}/results"                    # Prediction files of all models
os.makedirs(RESULTS, exist_ok=True)

# Copy the code (scripts/) and the data (data/) from Drive to Colab's local disk
for folder in ["scripts", "data"]:
    source = f"{DRIVE_DIR}/{folder}"
    assert os.path.exists(source), f"Missing on Drive: {source}"
    shutil.copytree(source, f"{PROJECT}/{folder}", dirs_exist_ok=True)

os.chdir(PROJECT)                                   # The scripts expect to be run from the project root
sys.path.insert(0, f"{PROJECT}/scripts")
print("Scripts:", sorted(os.listdir(f"{PROJECT}/scripts")))
print("Data folders:", sorted(os.listdir(f"{PROJECT}/data")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Scripts: ['.DS_Store', '__pycache__', 'baselines.py', 'diagnostics.py', 'download_data.py', 'evaluation.py', 'predictions.py', 'train_modernbert.py']
Data folders: ['preview', 'raw', 'splits']


In [11]:
import json, time, getpass, datetime
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import anthropic

# The project scripts are importable because the paths cell above added scripts/ to sys.path
from predictions import match_intent, write_predictions
import evaluation as ev

# Register the frontier model in the loaded module (also add it to scripts/evaluation.py on disk)
ev.MODEL_TYPE["frontier"] = "decoder"
print("Registered models:", [m for m in ev.MODEL_TYPE if "frontier" in m or "qwen" in m])

Registered models: ['qwen_zeroshot', 'qwen_qlora', 'qwen_sft', 'frontier']


#### API key and settings

The key is read from Colab Secrets

In [12]:
# Read the API key from Colab Secrets
if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

# Run settings
MODEL_ID = "claude-opus-5-5"     # API model string
MODEL_NAME = "frontier"           # Name used in the prediction files
BUDGET, SEED = "zeroshot", 0      # Zero-shot runs use seed 0 only
MAX_TOKENS = 2048                 # Opus 5.5 always thinks; thinking tokens count toward this limit
EFFORT = "low"                    # Thinking depth (None = API default, which is medium for Opus 5.5)
TEMPERATURE = 0                   # Deterministic as far as the API allows
WORKERS = 4                       # Parallel requests
CACHE_DIR = "reports/frontier_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Prices in USD per million tokens (Anthropic direct, standard rates)
PRICE_IN, PRICE_OUT = 4.0, 20.0

client = anthropic.Anthropic(max_retries=6)   # The SDK retries rate limits and transient errors

#### Backup to Drive

`backup()` copies the prediction files to the shared `results` folder on Drive, and the cache and run details to `frontier_backup`. It is called every 500 queries and after each stage. If the runtime resets, rerun the setup cells: this cell restores the cache and the run resumes without paying for finished queries again.

In [13]:
import glob

BACKUP = f"{DRIVE_DIR}/frontier_backup"             # Cache and run details
os.makedirs(BACKUP, exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("reports", exist_ok=True)

def backup():
    """Copy predictions to the shared results folder, and the cache and run details to the backup folder."""
    for f in glob.glob("results/frontier_*.csv"):
        shutil.copy(f, RESULTS)
    shutil.copytree(CACHE_DIR, f"{BACKUP}/frontier_cache", dirs_exist_ok=True)
    for f in glob.glob("reports/frontier_run_info.json"):
        shutil.copy(f, BACKUP)

# Restore from an earlier session, if there is one
if os.path.isdir(f"{BACKUP}/frontier_cache"):
    shutil.copytree(f"{BACKUP}/frontier_cache", CACHE_DIR, dirs_exist_ok=True)
for f in glob.glob(f"{RESULTS}/frontier_*.csv"):
    shutil.copy(f, "results/")
print("Backup folder:", BACKUP, "| Restored cache files:", len(glob.glob(f"{CACHE_DIR}/*.jsonl")))

Backup folder: /content/drive/MyDrive/banking77/frontier_backup | Restored cache files: 0


#### Data and prompt

The system line, the intent list and the message template are copied from the Qwen notebook (`Final_Decoder_Finetune.ipynb`), so Qwen zero-shot and the frontier model see exactly the same text.

In [14]:
# Load the label names, the validation set and the test set
labels = json.load(open("data/raw/label_names.json"))
val = pd.read_parquet("data/splits/val.parquet")
test = pd.read_parquet("data/raw/test.parquet")
print("Val:", val.shape, "| Test:", test.shape, "| Intents:", len(labels))

# Same prompt as the Qwen runs
SYSTEM = ("You classify banking customer messages. Reply with exactly one intent "
          "name from the list, copied exactly, and nothing else.")
INTENT_LIST = "\n".join(labels)

def user_text(message):
    return f"Intents:\n{INTENT_LIST}\n\nMessage: {message}\n\nIntent:"

print(user_text(val["text"].iloc[0])[-200:])

Val: (385, 3) | Test: (3080, 2) | Intents: 77
up
virtual_card_not_working
visa_or_mastercard
why_verify_identity
wrong_amount_of_cash_received
wrong_exchange_rate_for_cash_withdrawal

Message: Do you have info about the card on delivery?

Intent:


#### Classification helpers

Each query is one API call. Results are cached per example in a JSON-lines file, so a run can be stopped and resumed. Token counts are kept for the cost estimate.

In [15]:
USED_NO_TEMPERATURE = False   # Set to True if the API rejects the temperature setting

def call_model(message):
    """Send one query and return (raw_text, input_tokens, output_tokens, model_returned, stop_reason)."""
    global USED_NO_TEMPERATURE
    kwargs = dict(model=MODEL_ID, max_tokens=MAX_TOKENS, system=SYSTEM,
                  messages=[{"role": "user", "content": user_text(message)}])
    if EFFORT is not None:
        kwargs["output_config"] = {"effort": EFFORT}   # Only sent when you override the default
    if not USED_NO_TEMPERATURE:
        kwargs["temperature"] = TEMPERATURE
    try:
        resp = client.messages.create(**kwargs)
    except(anthropic.BadRequestError, TypeError) as e:
        # Some models reject sampling settings; retry once without temperature and note it
        if "temperature" in str(e).lower() and "temperature" in kwargs:
            USED_NO_TEMPERATURE = True
            print("Note: temperature was rejected, running with the API default instead")
            kwargs.pop("temperature", None)
            resp = client.messages.create(**kwargs)
        else:
            raise
    text = "".join(b.text for b in resp.content if b.type == "text").strip()
    return text, resp.usage.input_tokens, resp.usage.output_tokens, resp.model, getattr(resp, "stop_reason", None)

def classify(df, split, limit=None):
    """Classify every row of df (or the first `limit` rows), using and filling the cache."""
    path = f"{CACHE_DIR}/{split}.jsonl"
    cache = {}
    if os.path.exists(path):
        for line in open(path):
            r = json.loads(line)
            cache[r["example_id"]] = r
    ids = list(range(len(df) if limit is None else limit))
    todo = [i for i in ids if i not in cache]
    print(f"{split.capitalize()}: {len(ids) - len(todo)} cached, {len(todo)} to run")

    def work(i):
        text, tin, tout, mdl, stop = call_model(df["text"].iloc[i])
        return {"example_id": i, "raw": text, "in_tok": tin, "out_tok": tout, "model": mdl, "stop": stop}

    start = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool, open(path, "a") as fh:
        for n, r in enumerate(pool.map(work, todo), 1):
            cache[r["example_id"]] = r
            fh.write(json.dumps(r) + "\n"); fh.flush()
            if n % 100 == 0:
                print(f"Progress: {n}/{len(todo)} done, {time.time() - start:.0f} s")
            if n % 500 == 0 and "backup" in globals():
                backup()   # Copy progress to Drive so a runtime reset loses little
    return [cache[i] for i in ids]

def summarize(rows, df, split):
    """Match the raw outputs to intents and print accuracy, unmatched rate and cost."""
    raw = [r["raw"] for r in rows]
    pred = np.array([match_intent(t) for t in raw])
    y = df["label"].to_numpy()[:len(rows)]
    tin, tout = sum(r["in_tok"] for r in rows), sum(r["out_tok"] for r in rows)
    cost = tin / 1e6 * PRICE_IN + tout / 1e6 * PRICE_OUT
    cut = sum(1 for r in rows if r.get("stop") == "max_tokens")
    print(f"{split}: Cut off at max_tokens={cut} | Accuracy={(pred == y).mean():.3f} | Unmatched={(pred == -1).mean():.3f} | "
          f"Input tokens={tin:,} | Output tokens={tout:,} | Cost=${cost:.2f}")
    return raw, pred, tin, tout

#### Smoke test

Five validation examples. Check that the raw outputs are clean intent names and that the returned model string is the one we  expect. Stop here if anything looks wrong.

In [16]:
# Smoke-test calls are not cached, so they never mix with the real validation cache
smoke = []
for i in range(5):
    text, tin, tout, mdl, stop = call_model(val["text"].iloc[i])
    smoke.append((val["text"].iloc[i], text, labels[val["label"].iloc[i]], match_intent(text), mdl, stop, tout))
for msg, text, gold, p, mdl, stop, tout in smoke:
    print(f"Message: {msg}\n  Model output: {text!r} | Gold: {gold} | Matched index: {p} | Stop reason: {stop} | Output tokens: {tout}\n")
print("Model returned:", smoke[0][4])

Note: temperature was rejected, running with the API default instead
Message: Do you have info about the card on delivery?
  Model output: 'card_delivery_estimate' | Gold: card_arrival | Matched index: 12 | Stop reason: end_turn | Output tokens: 11

Message: How do I track the card you sent me?
  Model output: 'card_arrival' | Gold: card_arrival | Matched index: 11 | Stop reason: end_turn | Output tokens: 7

Message: Will I have to wait much longer for my new card to arrive?
  Model output: 'card_arrival' | Gold: card_arrival | Matched index: 11 | Stop reason: end_turn | Output tokens: 7

Message: I ordered a card and would like to know how to track the delivery progress of it.
  Model output: 'card_arrival' | Gold: card_arrival | Matched index: 11 | Stop reason: end_turn | Output tokens: 7

Message: Where is my new card if I haven't received it yet?
  Model output: 'card_arrival' | Gold: card_arrival | Matched index: 11 | Stop reason: end_turn | Output tokens: 7

Model returned: claud

#### Full validation run (385 examples)

In [17]:
val_rows = classify(val, "val")
val_raw, val_pred, val_tin, val_tout = summarize(val_rows, val, "Val")
write_predictions(MODEL_NAME, BUDGET, SEED, "val", val_pred, raw_outputs=val_raw)
backup()

Val: 0 cached, 385 to run
Progress: 100/385 done, 33 s
Progress: 200/385 done, 63 s
Progress: 300/385 done, 93 s
Val: Cut off at max_tokens=0 | Accuracy=0.894 | Unmatched=0.003 | Input tokens=311,263 | Output tokens=8,418 | Cost=$1.41


#### Full test run (3,080 examples)

Run this only after the validation numbers look sensible. The test set is used for the final number only.

In [18]:
test_rows = classify(test, "test")
test_raw, test_pred, test_tin, test_tout = summarize(test_rows, test, "Test")
write_predictions(MODEL_NAME, BUDGET, SEED, "test", test_pred, raw_outputs=test_raw)
backup()

Test: 0 cached, 3080 to run
Progress: 100/3080 done, 29 s
Progress: 200/3080 done, 61 s
Progress: 300/3080 done, 90 s
Progress: 400/3080 done, 118 s
Progress: 500/3080 done, 145 s
Progress: 600/3080 done, 173 s
Progress: 700/3080 done, 206 s
Progress: 800/3080 done, 233 s
Progress: 900/3080 done, 265 s
Progress: 1000/3080 done, 291 s
Progress: 1100/3080 done, 320 s
Progress: 1200/3080 done, 349 s
Progress: 1300/3080 done, 382 s
Progress: 1400/3080 done, 413 s
Progress: 1500/3080 done, 441 s
Progress: 1600/3080 done, 469 s
Progress: 1700/3080 done, 495 s
Progress: 1800/3080 done, 523 s
Progress: 1900/3080 done, 549 s
Progress: 2000/3080 done, 576 s
Progress: 2100/3080 done, 604 s
Progress: 2200/3080 done, 633 s
Progress: 2300/3080 done, 661 s
Progress: 2400/3080 done, 690 s
Progress: 2500/3080 done, 720 s
Progress: 2600/3080 done, 750 s
Progress: 2700/3080 done, 777 s
Progress: 2800/3080 done, 808 s
Progress: 2900/3080 done, 837 s
Progress: 3000/3080 done, 868 s
Test: Cut off at max_tok

#### Save the run details


In [19]:
info = {
    "model_requested": MODEL_ID,
    "models_returned": sorted({r["model"] for r in val_rows + test_rows}),
    "date_utc": datetime.datetime.now().isoformat(timespec="seconds"),
    "temperature": None if USED_NO_TEMPERATURE else TEMPERATURE,
    "effort": EFFORT or "default (medium)",
    "max_tokens": MAX_TOKENS,
    "input_tokens": val_tin + test_tin,
    "output_tokens": val_tout + test_tout,
    "cost_usd": round((val_tin + test_tin) / 1e6 * PRICE_IN + (val_tout + test_tout) / 1e6 * PRICE_OUT, 2),
    "val_unmatched": int((val_pred == -1).sum()),
    "test_unmatched": int((test_pred == -1).sum()),
}
os.makedirs("reports", exist_ok=True)
json.dump(info, open("reports/frontier_run_info.json", "w"), indent=2)
print(json.dumps(info, indent=2))
backup()

{
  "model_requested": "claude-opus-5-5",
  "models_returned": [
    "claude-opus-5-5"
  ],
  "date_utc": "2026-11-20T01:15:14",
  "temperature": null,
  "effort": "low",
  "max_tokens": 2048,
  "input_tokens": 2799856,
  "output_tokens": 76127,
  "cost_usd": 12.72,
  "val_unmatched": 1,
  "test_unmatched": 1
}
